## **Vector Search with minsearch**

Previous lessons code to generate the objects for this lesson

In [ ]:
# import the documents
from ingest import load_faq_data

documents = load_faq_data()

# embed the question and answer so the query can match against the question text and answer text in our index 
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

# import the model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# encode the embeddings in batch of 50
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

# turn them into 2 dimensional array(matrix) where rows are documents(vectors) and columns are dimensions of the vectors
import numpy as np
X = np.array(vectors)

In the previous section we did vector search by hand with numpy. We embedded the query, computed dot products, and found the best matches. Writing the argsort and matrix code every time gets old, and it can't filter by course. So instead we'll use a library that wraps all of it.

We'll use minsearch, the small in-memory search library we already used in module 1 for text search. It has a VectorSearch class for vector search.

Both classes share the same API:

* *fit* to index data   
* *search* to query     
* *filter_dict* in search to filter by keyword
  
It's the simplest way to get started with vector search.

## Creating the index

We already have our documents and vectors from the previous section.

Index them:

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

We pass the numpy array X with all embeddings and the list of documents as payload. The keyword_fields parameter works the same as in the text Index, so we can filter by course later.

## Searching
Let's search for a question:

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

Under the hood it does the same thing we just did by hand. It computes the dot product between each vector (after filtering) and our query vector.

Look at the top result:

In [ ]:
results[0]

It should return the document about joining the course late.

## Filtering by course

Like the text index, we can filter by keyword fields. This matters for user experience. A student in LLM Zoom Camp doesn't care about answers from the data engineering course. So we narrow to their course first, then score only within it.

Pass a *filter_dict*:

In [ ]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

Now that we can run vector search, let's use it in RAG.